# LeNet-5 on MNIST — Image Classification

**Question:** Implement LeNet-5 architecture for image classification on the MNIST dataset. Train the network and analyze classification performance.

---

## What is MNIST?
MNIST is a dataset of **70,000 grayscale images** of handwritten digits (0–9), each 28×28 pixels.
It's the classic "hello world" of image classification.

## What is LeNet-5?
LeNet-5 was invented by Yann LeCun in **1998** — one of the first successful CNNs ever built.
It was originally designed to read handwritten digits on bank cheques.

### Architecture (data flow)
```
Input (1 × 32 × 32)  ← grayscale image, resized from 28×28
  → Conv1: 5×5 filter, 1→6 channels     →  6 × 28 × 28
  → ReLU + AvgPool 2×2                  →  6 × 14 × 14
  → Conv2: 5×5 filter, 6→16 channels    → 16 × 10 × 10
  → ReLU + AvgPool 2×2                  → 16 ×  5 ×  5
  → Flatten                             →        400
  → FC1: 400 → 120, ReLU
  → FC2: 120 →  84, ReLU
  → FC3:  84 →  10  (one score per digit 0–9)
```

**Key differences from a plain CNN:**
- Uses **AvgPool** (average pooling) instead of MaxPool — original design choice from 1998
- Has **3 fully connected layers** instead of 2
- Originally used Sigmoid activations; we use **ReLU** (faster, works better)

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim          # renamed from 'opt' to avoid clash with the optimizer variable below
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

## Step 1 — Load the Data

MNIST images are 28×28, but LeNet-5 was designed for **32×32** input.
We use `Resize((32,32))` to upscale each image by 4 pixels on each side before feeding it to the network.

No normalization is applied here — pixel values stay in the range [0, 1] after `ToTensor()`.

In [ ]:
# Resize to 32×32 because LeNet-5 expects that input size
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# Download MNIST (60k training images, 10k test images)
train_dataset = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Batch the data — shuffle training so the model doesn't memorize order
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)  # BUG FIX: was using train_dataset here

## Step 2 — Define the LeNet-5 Model

The model has two parts:
1. **Feature extractor** — two conv+pool blocks that learn to detect edges, curves, and shapes
2. **Classifier** — three fully connected layers that map those features to digit probabilities

`torch.relu()` is applied directly inline instead of storing it as `self.relu = nn.ReLU()`. Both approaches do the same thing.

In [ ]:
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()

        # ── Feature Extractor ────────────────────────────────────────
        # Conv1: 1 input channel (grayscale), 6 output feature maps, 5×5 filter
        # No padding → 32×32 becomes 28×28 after the filter slides across
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6,  kernel_size=5)

        # Conv2: 6 → 16 feature maps, 5×5 filter
        # 14×14 input (after pool) → 10×10 output
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)

        # AvgPool halves spatial size: takes the average of each 2×2 block
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)

        # ── Classifier ───────────────────────────────────────────────
        # After conv2 + pool: 16 channels × 5×5 spatial = 400 values
        self.fc1 = nn.Linear(16 * 5 * 5, 120)  # 400 → 120
        self.fc2 = nn.Linear(120, 84)           # 120 → 84
        self.fc3 = nn.Linear(84, 10)            # 84  → 10 (one per digit class)

    def forward(self, x):
        # Block 1: Conv → ReLU → AvgPool
        x = self.pool(torch.relu(self.conv1(x)))   # 1×32×32  → 6×14×14

        # Block 2: Conv → ReLU → AvgPool
        x = self.pool(torch.relu(self.conv2(x)))   # 6×14×14  → 16×5×5

        # Flatten: convert 3D feature map to 1D vector
        # x.size(0) = batch size; -1 means "figure out the rest" = 16×5×5 = 400
        x = x.view(x.size(0), -1)                 # 16×5×5   → 400

        # Fully connected layers — ReLU on all except the final output layer
        x = torch.relu(self.fc1(x))               # 400 → 120
        x = torch.relu(self.fc2(x))               # 120 → 84
        x = self.fc3(x)                           # 84  → 10 (raw scores, no activation)
        return x

In [ ]:
model     = LeNet5()
criterion = nn.CrossEntropyLoss()  # Combines softmax + log loss; standard for classification
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Step 3 — Train the Model

We train for 5 **epochs** — one epoch = one full pass through all 60,000 training images.

Each iteration inside the loop:
1. `zero_grad()` — clear old gradients (they accumulate otherwise)
2. `model(images)` — forward pass: get predictions
3. `criterion(outputs, labels)` — measure how wrong the predictions are
4. `loss.backward()` — backprop: figure out which weights caused the error
5. `optimizer.step()` — adjust weights to reduce error

In [ ]:
for epoch in range(5):
    model.train()       # Enable training mode
    running_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()                    # Clear previous gradients
        outputs = model(images)                  # Forward pass
        loss = criterion(outputs, labels)        # Compute loss
        loss.backward()                          # Backpropagate
        optimizer.step()                         # Update weights
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/5 | Training Loss: {avg_loss:.4f}")

## Step 4 — Evaluate on the Test Set

We run the trained model on the **10,000 test images** it has never seen.

- `model.eval()` disables dropout/batchnorm training behavior (not used here, but good practice)
- `torch.no_grad()` tells PyTorch not to track gradients — saves memory and speeds up evaluation
- `torch.max(outputs, dim=1)` returns `(max_value, index)` — the index is the predicted class
  - e.g. if outputs = [0.1, 0.05, 0.8, ...], max index = 2, so the model predicts digit **2**

In [ ]:
model.eval()  # Switch to evaluation mode
correct = 0
total   = 0

with torch.no_grad():  # No gradient computation needed for evaluation
    for images, labels in test_loader:
        outputs = model(images)

        # argmax picks the class index with the highest score
        predicted = outputs.argmax(dim=1)

        total   += labels.size(0)                       # Count total images
        correct += (predicted == labels).sum().item()   # Count correct predictions

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Correctly classified {correct} out of {total} images")